We are using FIMO from the MEME Suite. It is installed as a Docker container and can be run in a specific directory with

docker run -v "$(pwd)":/home/meme -d memesuite/memesuite:latest

FIMO can then be run with in that container.

fimo --oc ./fimo_output --thresh 0.0001 Ray2013_rbp_Homo_sapiens.dna_encoded.meme single_designs.fa
fimo --oc ./fimo_output_mirna_targets --thresh 0.005 Ray2013_rbp_Homo_sapiens.dna_encoded.meme mirna_targets.fa

In [2]:
import pandas as pd
import os
import numpy
import scipy.stats as stats

## generate fasta file

In [3]:
# get mirbase
mirbase = pd.read_csv("../microrna_data/mirbase_extended.csv", index_col=0)

# get inserted designs
inserted_designs_1 = pd.read_csv("../design_files/inserted_designs/1_mirna_full_single_high_conf_inserted.csv", index_col=0)
inserted_designs_2 = pd.read_csv("../design_files/inserted_designs/2_mirna_full_single_low_conf_mirgenedb_inserted.csv", index_col=0)
inserted_designs = pd.concat([inserted_designs_1, inserted_designs_2])
inserted_designs = inserted_designs.set_index("miRNA1")

In [4]:
# save these as a fasta file
output_folder = "../FIMO"

with open(os.path.join(output_folder, "single_designs.fa"), "w") as f:
    for mirna in inserted_designs.index:
        f.write(f">{mirna}\n")
        f.write(f"{inserted_designs.loc[mirna, 'seq']}\n")
        
with open(os.path.join(output_folder, "mirna_targets.fa"), "w") as f:
    for mirna in mirbase.index:
        f.write(f">{mirna}\n")
        f.write(f"{mirbase.loc[mirna, 'target']}\n")

In [5]:
high_stability_mirnas = ['hsa-miR-365a-3p', 'hsa-miR-6802-3p', 'hsa-miR-3180-3p',
       'hsa-miR-582-5p', 'hsa-miR-197-3p', 'hsa-miR-9-3p',
       'hsa-miR-199a-5p', 'hsa-miR-342-3p', 'hsa-miR-340-5p',
       'hsa-miR-574-3p', 'hsa-miR-937-5p', 'hsa-miR-664b-3p',
       'hsa-miR-6791-5p', 'hsa-miR-3663-3p', 'hsa-miR-885-5p']
with open(os.path.join(output_folder, "high_stability_mirnas.fa"), "w") as f:
    for mirna in high_stability_mirnas:
        f.write(f">{mirna}\n")
        f.write(f"{mirbase.loc[mirna, 'target']}\n")

## Run FIMO

## Read in the resulting file

In [5]:
input_file = os.path.join(output_folder, "fimo_output_mirna_targets/fimo.tsv")
df_fimo = pd.read_csv(input_file, sep="\t")
#df_fimo.set_index("sequence_name", inplace=True)
df_fimo

,motif_id,motif_alt_id,sequence_name,start,stop,strand,score,p-value,q-value,matched_sequence
0,RNCMPT00072,SRSF2,hsa-miR-1825,3.0,10.0,+,11.71950,0.000012,0.0782,AGGAGAGG
1,RNCMPT00072,SRSF2,hsa-miR-6882-3p,9.0,16.0,+,11.71950,0.000012,0.0782,AGGAGAGG
2,RNCMPT00072,SRSF2,hsa-miR-483-3p,9.0,16.0,+,11.71950,0.000012,0.0782,AGGAGAGG
3,RNCMPT00072,SRSF2,hsa-miR-7109-3p,10.0,17.0,+,11.71950,0.000012,0.0782,AGGAGAGG
4,RNCMPT00072,SRSF2,hsa-miR-6823-3p,10.0,17.0,+,11.71950,0.000012,0.0782,AGGAGAGG
...,...,...,...,...,...,...,...,...,...,...
6289,RNCMPT00268,PTBP1,hsa-miR-1282,6.0,12.0,-,8.72789,0.000499,0.7340,TTTTTCT
6290,RNCMPT00268,PTBP1,hsa-miR-6832-3p,11.0,17.0,-,8.72789,0.000499,0.7340,TTTTTCT
6291,# FIMO (Find Individual Motif Occurrences): Ve...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6292,# The format of this file is described at http...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# load the deviation df
df_deviation = pd.read_csv("../outputs/3_fitting/combined_dataset/combined_dataset_deviation.csv", index_col=0)
df_deviation

,HEK293T,HeLa,SKNSH,MCF7,HUH7,A549,HaCaT,JEG3,Tera1,PC3
hsa-let-7a-3p,-0.014851,0.003325,-0.020337,0.026364,-0.022338,-0.001909,0.006001,-0.019002,-0.084692,0.064567
hsa-let-7a-5p,-0.045445,0.018718,0.187174,-0.255312,0.033477,-0.049017,-0.058934,0.142148,-0.119159,0.197415
hsa-let-7b-3p,-0.031168,-0.045728,-0.046873,-0.038836,-0.034085,-0.022010,-0.102620,-0.061370,-0.058173,-0.016885
hsa-let-7c-3p,0.009192,-0.011490,0.013342,-0.013654,0.010554,0.009805,-0.051490,-0.072508,-0.064482,-0.034341
hsa-let-7d-3p,0.029086,-0.147397,-0.071523,-0.140335,-0.000995,-0.048796,-0.224221,-0.014405,-0.031357,-0.185445
...,...,...,...,...,...,...,...,...,...,...
hsa-miR-892b,-0.020141,0.000671,0.000018,0.036663,-0.024445,-0.027480,-0.011594,-0.018966,0.006337,-0.017020
hsa-miR-934,-0.041725,-0.033330,-0.006969,-0.001306,-0.069197,-0.057521,-0.042116,-0.076929,-0.078366,-0.049338
hsa-miR-935,-0.011195,0.070744,-0.012866,0.024961,-0.004617,-0.004408,-0.027298,-0.018127,-0.028768,0.023749
hsa-miR-944,0.023993,0.024915,-0.027897,0.025565,0.041143,0.056149,0.034648,0.082210,0.071552,0.016295


In [28]:
# binarize FIMO by the miRNA
binary_fimo = df_fimo.groupby(['sequence_name', 'motif_alt_id']).size().unstack(fill_value=0)
binary_fimo[binary_fimo > 0] = 1  # Convert counts to binary

# Align binary_fimo to df_deviation
binary_fimo = binary_fimo.reindex(df_deviation.index).fillna(0)

In [53]:
# Initialize a DataFrame to store results
results = pd.DataFrame(columns=['Motif', 'Cell Line', 'Correlation Coefficient', 'P-value'])

# Iterate over each motif in binary_fimo
for motif in binary_fimo.columns:
    # Iterate over each cell line in df_deviation
    for cell_line in df_deviation.columns:
        # calculate Pearson correlation and p-value
        df_deviation_cell_line = df_deviation[cell_line].dropna()
        corr, p_value = stats.pearsonr(binary_fimo.loc[df_deviation_cell_line.index, motif], df_deviation_cell_line)

        # Append results
        new_row = pd.DataFrame({
            'Motif': motif,
            'Cell Line': cell_line,
            'Correlation Coefficient': corr,
            'P-value': p_value
        }, index=[motif+"_"+cell_line])
        
        results = pd.concat([results, new_row])
       
significant_results = results[results['P-value'] < 0.05]
significant_results = significant_results.sort_values(by="P-value", ascending=True)
print(significant_results) 

C:\Users\loesi\AppData\Local\Temp\ipykernel_11004\3130456616.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row])
c:\Users\loesi\anaconda3\envs\Bio\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
c:\Users\loesi\anaconda3\envs\Bio\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
c:\Users\loesi\anaconda3\envs\Bio\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correla

<bound method DataFrame.sort_values of                    Motif Cell Line  Correlation Coefficient       P-value
ZNF638_MCF7       ZNF638      MCF7                 0.190442  1.160485e-09
YBX1_SKNSH          YBX1     SKNSH                 0.157476  4.972408e-07
PTBP1_Tera1        PTBP1     Tera1                -0.141989  6.225189e-06
PTBP1_SKNSH        PTBP1     SKNSH                -0.136746  1.307643e-05
KHDRBS1_HeLa     KHDRBS1      HeLa                -0.132556  2.514072e-05
ZNF638_HUH7       ZNF638      HUH7                 0.108219  5.671970e-04
KHDRBS1_HUH7     KHDRBS1      HUH7                -0.101638  1.211837e-03
KHDRBS1_MCF7     KHDRBS1      MCF7                -0.095050  2.558636e-03
SNRPA_MCF7         SNRPA      MCF7                -0.088636  4.924104e-03
KHDRBS1_SKNSH    KHDRBS1     SKNSH                -0.088013  5.147101e-03
KHDRBS1_A549     KHDRBS1      A549                -0.080560  1.050709e-02
PABPC5_JEG3       PABPC5      JEG3                 0.080445  1.101314e-02

In [59]:
df_deviation['mean_deviation'] = df_deviation.fillna(0).mean(axis=1)

# Initialize a DataFrame to store results
results = pd.DataFrame(columns=['Motif', 'Correlation Coefficient', 'P-value'])

# Iterate over each motif in binary_fimo
for motif in binary_fimo.columns:
    # Calculate Pearson correlation and p-value between the motif and the aggregate deviation
    corr, p_value = stats.pearsonr(binary_fimo[motif], df_deviation['mean_deviation'])

    # Append results
    new_row = pd.DataFrame({
        'Motif': motif,
        'Correlation Coefficient': corr,
        'P-value': p_value
    }, index=[motif])
    
    results = pd.concat([results, new_row])

# Optionally, filter results to show only statistically significant correlations
significant_results = results[results['P-value'] < 0.05]
significant_results = significant_results.sort_values(by="P-value", ascending=True)
print(significant_results)

           Motif  Correlation Coefficient   P-value
KHDRBS1  KHDRBS1                -0.086278  0.006050
PTBP1      PTBP1                -0.078573  0.012451
ZNF638    ZNF638                 0.070279  0.025443


C:\Users\loesi\AppData\Local\Temp\ipykernel_11004\3769256464.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row])
c:\Users\loesi\anaconda3\envs\Bio\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [63]:
for RBP in significant_results.index:
    RBP_mirnas = df_fimo[df_fimo["motif_alt_id"] == RBP]["sequence_name"]
    print(RBP)
    print(df_deviation.loc[RBP_mirnas, :])

KHDRBS1
                 HEK293T      HeLa     SKNSH      MCF7      HUH7      A549  \
hsa-miR-590-5p -0.170999 -0.303581 -0.226934 -0.235054 -0.215432 -0.197553   

                   HaCaT      JEG3     Tera1       PC3  mean_deviation  
hsa-miR-590-5p -0.168665 -0.126686 -0.058606 -0.117463       -0.182097  
PTBP1
                   HEK293T      HeLa     SKNSH      MCF7      HUH7      A549  \
hsa-miR-580-5p   -0.033675  0.011420 -0.042002  0.019259  0.038394  0.046839   
hsa-miR-518c-5p  -0.012345  0.031068  0.011084  0.025521 -0.048714  0.002491   
hsa-miR-526b-5p  -0.001271 -0.003798  0.000932  0.040436 -0.036315 -0.008425   
hsa-miR-520a-5p  -0.012686  0.012034 -0.007832  0.016401  0.028335  0.028158   
hsa-miR-518f-5p  -0.016566  0.007206  0.006028  0.034310 -0.022454  0.019048   
hsa-miR-548x-3p   0.022763  0.028153  0.028313  0.000594  0.023828  0.041227   
hsa-miR-525-5p   -0.054922 -0.035622 -0.016691  0.021360 -0.057648 -0.020439   
hsa-miR-520g-5p  -0.033884  0.007404  0.001